In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# --- KONFIGURACJA ---
REF_MASK_PATH = "reference_mask_vote_0.50.npy"
BRIGHTNESS_THRESHOLD = 105  # Im niżej, tym czulszy na szare rysy
MIN_DAMAGE_AREA = 120      # Ile pikseli błędu uznajemy za wadę
EROSION_SIZE = 5           # Margines od krawędzi (bezpieczeństwo)

# Załaduj maskę referencyjną
if os.path.exists(REF_MASK_PATH):
    REFERENCE_MASK = np.load(REF_MASK_PATH).astype(np.uint8)
    if REFERENCE_MASK.max() == 1: REFERENCE_MASK *= 255
    print("Maska referencyjna załadowana.")
else:
    print("BŁĄD: Nie znaleziono pliku maski referencyjnej!")

In [ ]:
def get_aligned_mask(test_mask, ref_mask):
    """Proste wyrównanie testowej maski do wzorca na podstawie środka ciężkości."""
    M_ref = cv2.moments(ref_mask)
    M_test = cv2.moments(test_mask)
    
    if M_ref["m00"] == 0 or M_test["m00"] == 0:
        return test_mask # Nie można wyrównać
        
    dx = int(M_ref["m10"]/M_ref["m00"] - M_test["m10"]/M_test["m00"])
    dy = int(M_ref["m01"]/M_ref["m00"] - M_test["m01"]/M_test["m00"])
    
    rows, cols = test_mask.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(test_mask, M, (cols, rows), flags=cv2.INTER_NEAREST)

def extract_mask(img_rgb):
    """Uproszczona ekstrakcja tranzystora (możesz tu wstawić funkcję Piotra)."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    # Próg 60 zazwyczaj dobrze odcina czarny plastik od miedzi
    _, mask = cv2.threshold(gray, 60, 255, cv2.THRESH_BINARY_INV)
    # Czyszczenie drobnych śmieci
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask

In [ ]:
def detect_damaged_case(img_rgb, ref_mask):
    # 1. Przygotowanie masek i obrazu
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    test_mask = extract_mask(img_rgb)
    
    # 2. Wyrównanie testowej maski do wzorca
    test_mask_aligned = get_aligned_mask(test_mask, ref_mask)
    
    # 3. DETEKCJA KSZTAŁTU (Shape Check) - szukamy ubytków
    # Czego brakuje w teście względem wzorca?
    shape_defect = cv2.subtract(ref_mask, test_mask_aligned)
    
    # 4. DETEKCJA JASNOŚCI (Brightness Check) - szukamy rys i pęknięć
    # Robimy bezpieczny margines (erozja), żeby ignorować błyszczące krawędzie
    kernel = np.ones((EROSION_SIZE, EROSION_SIZE), np.uint8)
    safe_zone = cv2.erode(ref_mask, kernel, iterations=1)
    
    _, bright_parts = cv2.threshold(gray, BRIGHTNESS_THRESHOLD, 255, cv2.THRESH_BINARY)
    brightness_defect = cv2.bitwise_and(bright_parts, safe_zone)
    
    # 5. ŁĄCZENIE WAD I FINALNA DECYZJA
    combined_defect = cv2.bitwise_or(shape_defect, brightness_defect)
    
    # Usuwamy szum (pojedyncze piksele)
    combined_defect = cv2.morphologyEx(combined_defect, cv2.MORPH_OPEN, kernel)
    
    defect_area = np.count_nonzero(combined_defect)
    is_damaged = defect_area > MIN_DAMAGE_AREA
    
    return is_damaged, combined_defect, test_mask_aligned

In [ ]:
# Załaduj przykładowe zdjęcie (zmień ścieżkę!)
test_img_path = "transistor/test/damaged_case/000.png" 

if os.path.exists(test_img_path):
    img = cv2.cvtColor(cv2.imread(test_img_path), cv2.COLOR_BGR2RGB)
    
    is_def, defect_map, aligned_m = detect_damaged_case(img, REFERENCE_MASK)
    
    # --- Wyświetlanie ---
    plt.figure(figsize=(16, 6))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img)
    plt.title(f"Oryginał\nWynik: {'USZKODZONY' if is_def else 'OK'}")
    
    plt.subplot(1, 4, 2)
    plt.imshow(aligned_m, cmap='gray')
    plt.title("Wyrównana Maska Testowa")
    
    plt.subplot(1, 4, 3)
    plt.imshow(defect_map, cmap='hot')
    plt.title(f"Mapa Defektów\n(Piksele: {np.count_nonzero(defect_map)})")
    
    # Nakładka błędu na oryginał
    overlay = img.copy()
    overlay[defect_map > 0] = [255, 0, 0] # Czerwony kolor błędu
    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.title("Lokalizacja uszkodzenia")
    
    plt.tight_layout()
    plt.show()
else:
    print("Brak pliku testowego.")